# 068 — The Encoder–Decoder Architecture

Two LSTMs facing each other: the encoder reads the input and hands over its
final state, the decoder starts from that state and emits tokens until it
produces `<END>`. Three details make it work — teacher forcing, the
`<START>`/`<END>` tokens, and separate weights — and one undoes it: **the
handover is a fixed 2,000 numbers**, which is 400 per token for a short
sentence and **20** for a long one.

| Part | What we check |
|---|---|
| A | the handover, and how output length gets decided |
| B | teacher forcing against feeding back predictions |
| C | the bottleneck as arithmetic |
| D | why it is not a raw-storage limit |

## Part A — Encode, hand over, decode

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

F, H = 8, 20
rng = np.random.default_rng(0)
Wenc = {g: rng.normal(0, 0.3, (F + H, H)) for g in "fico"}
Wdec = {g: rng.normal(0, 0.3, (F + H, H)) for g in "fico"}   # SEPARATE weights

def lstm_step(W, x, h, C):
    z = np.concatenate([h, x])
    f = sigmoid(z @ W["f"])
    i = sigmoid(z @ W["i"])
    cand = np.tanh(z @ W["c"])
    C = f * C + i * cand
    return sigmoid(z @ W["o"]) * np.tanh(C), C

# encode a 7-token input, emitting nothing
source = rng.normal(0, 1, (7, F))
h, C = np.zeros(H), np.zeros(H)
for x in source:
    h, C = lstm_step(Wenc, x, h, C)
context = (h.copy(), C.copy())
print(f"  encoded {len(source)} tokens -> context of {2 * H} numbers "
      f"(h {H} + C {H})")

# decode until <END>, which here is "when the first unit goes negative"
START, decoded = np.zeros(F), []
x, h, C = START, *context
for step in range(10):
    h, C = lstm_step(Wdec, x, h, C)
    decoded.append(h.copy())
    if h[0] < 0:                       # stands in for predicting <END>
        break
    x = h[:F]
print(f"  decoded {len(decoded)} tokens from a {len(source)}-token input")
assert len(decoded) != len(source)
print("  The lengths differ, and nothing was told what the output length should")
print("  be: the <END> token decides it at run time. That is the whole point.")

## Part B — Teacher forcing

During training the decoder is fed the **correct** previous token; at inference
it is fed its own prediction. Simulate both with a decoder that drifts.

In [ ]:
truth = ["le", "chat", "est", "noir", "<END>"]
wrong_at = 1                                 # the decoder slips on token 2

def decode(teacher_forcing):
    out, drifted = [], False
    for t, want in enumerate(truth):
        if drifted and not teacher_forcing:
            out.append("<drift>")            # errors compound
            continue
        got = want if t != wrong_at else "chien"
        out.append(got)
        if got != want:
            drifted = True
    return out

print("  teacher forcing (training) :", decode(True))
print("  feeding back (inference)   :", decode(False))
tf, fb = decode(True), decode(False)
assert tf.count("<drift>") == 0 and fb.count("<drift>") > 0
print()
print("With teacher forcing, one mistake costs one token: every other step still")
print("starts from the truth. Without it, one mistake poisons the rest of the")
print("sequence - which is why training and inference differ here, and why that")
print("mismatch has a name: exposure bias.")

## Part C — The bottleneck, as arithmetic

In [ ]:
H_PAPER = 1000                       # the 2014 model's hidden size
context_width = 2 * H_PAPER          # final h and C

print(f"  {'sentence length':>16}{'context vector':>16}{'numbers per token':>20}")
for length in (5, 10, 20, 50, 100):
    print(f"  {length:>16}{context_width:>16,}{context_width / length:>20.0f}")
assert context_width == 2000
assert context_width / 5 == 400 and context_width / 100 == 20
print()
print("The first column grows and the second never does. Whatever the decoder")
print("needs to know about token 3 of a 100-token sentence competes for the")
print("same 2,000 numbers as everything else: 400 per token becomes 20.")

## Part D — But not for the reason usually given

It is tempting to say 2,000 numbers cannot *hold* a long sentence. Check that
claim before repeating it.

In [ ]:
import math

bits_available = 2000 * 32                        # float32
sentence_bits = 20 * math.log2(80_000)            # 20 tokens, 80k vocabulary
print(f"  2,000 float32 values hold        {bits_available:>10,} bits")
print(f"  identifying a 20-token sentence  {sentence_bits:>10,.0f} bits")
print(f"  ratio                            {bits_available / sentence_bits:>10,.0f}x")
assert bits_available > sentence_bits * 100
print()
print("Raw capacity is not the constraint - it is off by a factor of ~196. The")
print("binding constraint is that the vector must be produced by a smooth")
print("recurrent map and consumed by another, so nearby sentences must land on")
print("nearby vectors for training to work at all. That continuity requirement")
print("caps usable capacity far below the bit count.")
print()
print("The paper's own trick is the admission: REVERSING the source sentence")
print("helps, which would make no difference if the summary were lossless.")

## What this notebook established

- The encoder emits nothing and hands over a fixed context — $2H$ numbers, the
  final $h$ and $C$ — and the decoder's `<END>` token decides the output length
  at run time, so input and output lengths need not match.
- Teacher forcing keeps one mistake local; feeding predictions back lets one
  mistake poison the rest of the sequence.
- The context is **2,000 numbers** in the 2014 model: **400** per token for a
  5-token sentence, **20** for a 100-token one.
- It is **not** a raw-storage limit — 2,000 float32 values hold about 196x the
  bits needed to identify a 20-token sentence. The limit is the continuity the
  recurrent map requires, and the paper's source-reversal trick is the
  admission.

## Exercises

1. Make the decoder's `<END>` condition stricter and watch the output length
   change. What happens if it never fires?
2. Average every encoder state instead of taking the last. Does that fix the
   bottleneck or move it?
3. Hand the context to **every** decoder step rather than only the first. You
   have just invented half of attention — what is still missing?